# Setup

In [ ]:
%run common.py
import sys
sys.path.append("../../legal-data-clustering/")
%run '../../legal-data-clustering/legal_data_clustering/utils/graph_api.py'

In [ ]:
de_crossreference_path = f'../../legal-networks-data/de/4_crossreference_graph/seqitems'

In [ ]:
# xml_files = {(f.split('_')[0], os.path.splitext(f)[0].split('_')[3]):f for f in list_dir('../../legal-networks-data/de/2_xml/', '.xml')}
# def get_text_for_node(node):
#     
#     dknr, abk, end_date, item_idx = os.path.splitext(node)[0].split('_')
#     file = xml_files[(dknr,end_date)]
#     with open(f'../../legal-networks-data/de/2_xml/{file}') as f:
#         soup = BeautifulSoup(f.read())
#     return soup.find(key=node).get_text()

In [ ]:
def format_head(G, nodes, cut=10):
    return [
        f'{cnt:4}' + ' ' + G.nodes[n]['law_name'] + ': ' + G.nodes[n]['heading']
        for n, cnt in nodes[:cut]
    ]

def get_all_reference_edges_graph(G):
    return induced_subgraph(G, 'edge', 'edge_type', ['reference'])

def get_references_other_law_title(G: nx.MultiDiGraph):
    return [ 
        (u, v, k, data)
        for u, v, k, data 
        in G.edges(data=True, keys=True)
        if data['edge_type'] == "reference" and u.split('_')[0] != v.split('_')[0]
    ]


def get_references_other_law_title_graph(G: nx.MultiDiGraph):
    sG = nx.MultiDiGraph()
    sG.add_nodes_from(G.nodes(data=True))
    sG.add_edges_from(get_references_other_law_title(G))
    return sG

In [ ]:
def select_key_parts(data, pos):
    return [
        (
            n.split('_')[pos],
            cnt,
        )
        for n, cnt in data
    ]

def select_human_readable_citekey(data, G):
    return [
        (
            ' '.join(G.nodes[n]['heading'].split(' ')[:2]) + ' ' + G.nodes[n]['citekey'].split('_')[0],
            cnt
        )
        for n, cnt in data
    ]

def select_citekey(data, abk_units):
    for citekey, cnt in data:
        abk, nr = citekey.split('_')
        try:
            unit = abk_units[abk]
        except KeyError:
            print(abk, 'not in abk_units')
            unit = '§'
        yield (
            f'{unit} {nr} {abk}',
            cnt,
        )

def save_tex_from_lists(data, columns, filename):
    '''
    Converts a list of Counter.most_common results to a latex table.
    '''
    tex = '\\begin{tabular}{rl' + '|rl' * (len(data)-1) + '}\n'
    if columns:
        tex += '\multicolumn{2}{c}{' + str(columns[0]) + '}' 
    tex += ''.join(
        ' & \multicolumn{2}{|c}{' + str(column) + '}' 
        for column in columns[1:]
    )
    tex += ' \\\\\n'
    tex += ' & '.join(
        'Zitate & Norm' 
        for column in columns
    )
    tex += ' \\\\\n'
    tex += '\\midrule\n'
    

    for row_idx in range(len(data[0])):
        
        row_tup = [
            data[col_idx][row_idx] for col_idx in range(len(data))
        ]
        tex += ' & '.join(de_num_format(f'{cnt:,}') + f' & {name}' for name, cnt in row_tup)
        tex += ' \\\\\n'
#     tex += '\\bottomrule\n'
    tex += '\\end{tabular}\n'


    with open(f'../tables/{filename}.tex', 'w') as f:
        f.write(tex)
        return tex

## Crossreference

In [ ]:
de_graph_files = sorted(list_dir(de_crossreference_path, 'gpickle.gz'))[-1:]
de_graphs = [nx.read_gpickle(f'{de_crossreference_path}/{gf}') for gf in de_graph_files]

In [ ]:
G_crossreference = de_graphs[-1]
G_seqitems = G_crossreference.subgraph([n for n, t in G_crossreference.nodes(data='type') if t == 'seqitem']).copy()

G_seqitems_binary = make_weighted(G_seqitems)
G_seqitems_other_law = get_references_other_law_title_graph(G_seqitems)
G_seqitems_other_law_binary = make_weighted(G_seqitems_other_law)


nx.set_node_attributes(G_seqitems, {n: '_'.join(n.split('_')[:3]) for n in G_seqitems.nodes}, 'merge_attr')
G_documents = quotient_graph(G_seqitems, 'merge_attr')
G_documents_binary = make_weighted(G_documents)

In [ ]:
counter_art = defaultdict(int)
counter_para= defaultdict(int)

for node, heading in nx.get_node_attributes(G_seqitems, 'heading').items():
    abk = node.split('_')[1]
    if heading.startswith('§'):
        counter_para[abk] += 1
    elif heading.lower().startswith('art'):
        counter_art[abk] += 1
        
abk_units = {
    abk: ('§' if counter_para[abk] > counter_art[abk] else 'Art')
    for abk in (set(counter_art) | set(counter_para))
}

## Citing lists

In [ ]:
top_citing_seqitem = sorted(G_seqitems.out_degree(), reverse=True, key=lambda x: x[1])
top_citing_seqitem_other_law = sorted(G_seqitems_other_law.out_degree(), reverse=True, key=lambda x: x[1])

print(save_tex_from_lists(
    data=[
        select_human_readable_citekey(top_citing_seqitem[:25], G_seqitems), 
        select_human_readable_citekey(top_citing_seqitem_other_law[:25], G_seqitems_other_law),
    ],
    columns=['Alle Querverweise' , 'Querverweise zu anderen Gesetzen'],
    filename='mikro_top_citing_seqitem_de'
))

In [ ]:
top_citing_documents = sorted(G_documents.out_degree(), reverse=True, key=lambda x: x[1])
top_citing_documents_binary = sorted(G_documents_binary.out_degree(), reverse=True, key=lambda x: x[1])

print(save_tex_from_lists(
    data=[
        select_key_parts(top_citing_documents[:25], pos=1), 
        select_key_parts(top_citing_documents_binary[:25], pos=1),
    ],
    columns=['Gewichtet' , 'Binär'],
    filename='mikro_top_citing_documents_de'
))

## Cited lists

In [ ]:
top_cited_seqitem = sorted(G_seqitems.in_degree(), reverse=True, key=lambda x: x[1])
top_cited_seqitem_binary = sorted(G_seqitems_binary.in_degree(), reverse=True, key=lambda x: x[1])
top_cited_seqitem_other_law = sorted(G_seqitems_other_law.in_degree(), reverse=True, key=lambda x: x[1])
top_cited_seqitem_other_law_binary = sorted(G_seqitems_other_law_binary.in_degree(), reverse=True, key=lambda x: x[1])

print(save_tex_from_lists(
    data=[
        select_human_readable_citekey(top_cited_seqitem_binary[:25], G_seqitems_binary), 
        select_human_readable_citekey(top_cited_seqitem_other_law_binary[:25], G_seqitems_other_law_binary),
    ],
    columns=['Gewichtet' , 'Binär und von anderen Gesetzen'],
    filename='mikro_top_cited_seqitem_de'
))

In [ ]:
top_cited_documents = sorted(G_documents.in_degree(), reverse=True, key=lambda x: x[1])
top_cited_documents_binary = sorted(G_documents_binary.in_degree(), reverse=True, key=lambda x: x[1])

print(save_tex_from_lists(
    data=[
        select_key_parts(top_cited_documents[:25], pos=1), 
        select_key_parts(top_cited_documents_binary[:25], pos=1),
    ],
    columns=['Gewichtet' , 'Binär'],
    filename='mikro_top_cited_documents_de'
))

# Loading

In [ ]:
G = nx.read_gpickle('../../legal-networks-data/de_decisions/2_network.gpickle.gz')

In [ ]:
propagate_attrs_to_descendents(G, ['gericht', 'spruchkoerper', 'datum'])

In [ ]:
G_ms = quotient_decision_graph(G, merge_decisions=False, merge_statutes=True)
G_md = quotient_decision_graph(G, merge_decisions=True, merge_statutes=False)
G_md_ms = quotient_decision_graph(G, merge_decisions=True, merge_statutes=True)

# Zitate in Gerichtsentscheidungen

In [ ]:
def get_in_degree(G, n, weighted, gericht=None):
    edge_weights = [
        w 
        for u, v, w in G.in_edges(n, data='weight')
        if gericht is None or G.nodes[u]['gericht'] == gericht
    ]
    if weighted:
        return sum(edge_weights)
    else:
        return len(edge_weights)

def highlight_highest_rank(data):
    name_rank  = defaultdict(list)
    data = [list(d) for d in data]
    for col in data:
        for idx, (name, cnt) in enumerate(col):
            name_rank[name].append(idx)
    top_name_rank = {
        name: min(ranks) for name, ranks in name_rank.items()
    }
    return [
        [
            (
                ('\\emph{'+name+'}' if top_name_rank[name] == idx else name),
                cnt,
            )
            for idx, (name, cnt) in enumerate(col)
        ]
        for col in data
    ]

In [ ]:
top_cited_seqitem_by_decision = sorted([(n, get_in_degree(G, n, weighted=True)) for n, b in G.nodes(data='bipartite') if b == 'statute'], key=lambda x: -x[1])[:25]
top_cited_seqitem_by_decision_para_binary = sorted([(n, get_in_degree(G, n, weighted=False)) for n, b in G.nodes(data='bipartite') if b == 'statute'], key=lambda x: -x[1])[:25]
top_cited_seqitem_by_decision_binary = sorted([(n, get_in_degree(G_md, n, weighted=False)) for n, b in G_md.nodes(data='bipartite') if b == 'statute'], key=lambda x: -x[1])[:25]

In [ ]:
print(save_tex_from_lists(
    data=highlight_highest_rank(
        [
        select_citekey(top_cited_seqitem_by_decision, abk_units),
#         select_citekey(top_cited_seqitem_by_decision_para_binary, abk_units),
        select_citekey(top_cited_seqitem_by_decision_binary, abk_units),
    ]
    )
    ,
    columns=[
        'Gewichtet' , 
#         'Binär für Absätze', 
        'Binär für Entscheidungen'
    ],
    filename='mikro_top_cited_seqitems_decisions_de'
))

In [ ]:
def shorten_label(item_list, max_len=15):
    res = []
    for item in item_list:
        label, cnt = item
        if len(label) > max_len:
            label = re.sub(r"-\d{4}", "...", label)
        res.append((label, cnt))
    return res

def top_lists_to_chart(top_lists_formatted, gerichte, separation_factor=0.055, domain_min_factor=0.95, domain_max_factor=1.08):
    global df
    df = pd.DataFrame(
        [
            (gericht or 'Alle', name, cnt)
            for gericht, top_list in zip(gerichte, top_lists_formatted)
            for name, cnt in top_list
            if gericht != 'GmSOGB'
        ],
        columns=['Gericht','Norm','Zitate']
    )

    zit_korr_all = []
    max_limit, min_limit = df.Zitate.max(), df.Zitate.min()                

    for gericht in df.Gericht.unique():
        zit_korr = df[df.Gericht==gericht].Zitate.to_list()
        corrected = True
        i = 0
        while corrected:
            i += 1
            corrected = False
            for idx in range(len(zit_korr)-1):            
                min_dist =  (zit_korr[idx] + zit_korr[idx+1]) / 2 * separation_factor
                dist = zit_korr[idx] - zit_korr[idx+1]
                if dist < min_dist:
                    if zit_korr[idx] < max_limit:
                        zit_korr[idx] += 1
                    if zit_korr[idx+1] > min_limit:
                        zit_korr[idx+1] -= 1
                    
                    corrected = True
            if i % 10e4 == 0:
                print(gericht, i)
            if i > 10e6:
                print('limit reached')
                break
        zit_korr_all.extend(zit_korr)
    df['Zitate_korr'] =  zit_korr_all



    chart = alt.Chart(df).mark_point(size=10, xOffset=(-32 if len(df.Gericht.unique()) > 7 else -35), opacity=0.5, color="black", filled=True).encode(
        alt.X('Gericht:N', scale=alt.Scale(padding=0.5)),
        alt.Y('Zitate:Q', scale=alt.Scale(type='log', nice=False,domain=(df.Zitate.min()*domain_min_factor,df.Zitate.max()*domain_max_factor ) )),    
    )

    chart += alt.Chart(df).mark_text(align='left', xOffset=(-29 if len(df.Gericht.unique()) > 7 else -32), limit=100).encode(
        alt.X('Gericht:N', axis=alt.Axis(grid=False, labelAngle=0, labelAlign='center', labelPadding=10), title=None),
        alt.Y('Zitate_korr:Q', title=None, axis=alt.Axis(grid=False)),
        alt.Text('Norm'),
    )

    chart = chart.properties(width=600, height=335)
    return chart

In [ ]:
gerichte = sorted({g for n, g in G.nodes(data='gericht') if g})

In [ ]:
top_lists = [
    sorted(
        [
            (n, get_in_degree(G_md, n, weighted=False, gericht=gericht)) 
            for n, b in G_md.nodes(data='bipartite') 
            if b == 'statute'
        ], 
        key=lambda x: -x[1]
    )[:25]
    for gericht in gerichte
]

top_lists_formatted = [shorten_label(select_citekey(l, abk_units)) for l in top_lists]
chart = top_lists_to_chart(top_lists_formatted, gerichte, separation_factor=0.11)
used_abks = {label.split('_')[0] for gericht_list in top_lists for label, cnt in gericht_list}
save_chart(chart, 'mikro_top_cited_seqitems_decisions_de_gerichte_decision_binary', used_abks)

In [ ]:
top_lists = [
    sorted(
        [
            (n, get_in_degree(G, n, weighted=True, gericht=gericht)) 
            for n, b in G.nodes(data='bipartite') 
            if b == 'statute'
        ], 
        key=lambda x: -x[1]
    )[:25]
    for gericht in gerichte
]

top_lists_formatted = [shorten_label(select_citekey(l, abk_units)) for l in top_lists]
chart = top_lists_to_chart(top_lists_formatted, gerichte, separation_factor=0.14)
used_abks = {label.split('_')[0] for gericht_list in top_lists for label, cnt in gericht_list}
save_chart(chart, 'mikro_top_cited_seqitems_decisions_de_gerichte', used_abks)

In [ ]:
top_lists = [
    sorted(
        [
            (n, get_in_degree(G, n, weighted=False, gericht=gericht)) 
            for n, b in G.nodes(data='bipartite') 
            if b == 'statute'
        ], 
        key=lambda x: -x[1]
    )[:25]
    for gericht in gerichte
]

top_lists_formatted = [shorten_label(select_citekey(l, abk_units)) for l in top_lists]
chart = top_lists_to_chart(top_lists_formatted, gerichte, separation_factor=0.12)
used_abks = {label.split('_')[0] for gericht_list in top_lists for label, cnt in gericht_list}
save_chart(chart, 'mikro_top_cited_seqitems_decisions_de_gerichte_abs_binary', used_abks)

In [ ]:
top_lists = [
    sorted(
        [
            (n, get_in_degree(G_ms, n, weighted=True, gericht=gericht)) 
            for n, b in G_ms.nodes(data='bipartite') 
            if b == 'statute'
        ], 
        key=lambda x: -x[1]
    )[:25]
    for gericht in [None, *gerichte]
]

top_lists_formatted = [shorten_label(l, max_len=12) for l in top_lists]
chart = top_lists_to_chart(top_lists_formatted, [None, *gerichte], separation_factor=0.25, domain_min_factor=0.88, domain_max_factor=1.17)
used_abks = {label.split('_')[0] for gericht_list in top_lists for label, cnt in gericht_list}
save_chart(chart, 'mikro_top_cited_documents_decisions_de_gerichte', used_abks)

In [ ]:
top_lists = [
    sorted(
        [
            (n, get_in_degree(G_ms, n, weighted=False, gericht=gericht)) 
            for n, b in G_ms.nodes(data='bipartite') 
            if b == 'statute'
        ], 
        key=lambda x: -x[1]
    )[:25]
    for gericht in [None, *gerichte]
]

top_lists_formatted = [shorten_label(l, max_len=12) for l in top_lists]
chart = top_lists_to_chart(top_lists_formatted, [None, *gerichte], separation_factor=0.22, domain_min_factor=0.88, domain_max_factor=1.17)
used_abks = {label.split('_')[0] for gericht_list in top_lists for label, cnt in gericht_list}
save_chart(chart, 'mikro_top_cited_documents_decisions_de_gerichte_abs_binary', used_abks)

In [ ]:
top_lists = [
    sorted(
        [
            (n, get_in_degree(G_md_ms, n, weighted=False, gericht=gericht)) 
            for n, b in G_md_ms.nodes(data='bipartite') 
            if b == 'statute'
        ], 
        key=lambda x: -x[1]
    )[:25]
    for gericht in [None, *gerichte]
]

top_lists_formatted = [shorten_label(l, max_len=12) for l in top_lists]
chart = top_lists_to_chart(top_lists_formatted, [None, *gerichte], separation_factor=0.24, domain_min_factor=0.88, domain_max_factor=1.17)
used_abks = {label.split('_')[0] for gericht_list in top_lists for label, cnt in gericht_list}
save_chart(chart, 'mikro_top_cited_documents_decisions_de_gerichte_decision_binary', used_abks)

## Analyse ad-hoc

In [ ]:
[
    f'{cnt:4} - {n} '+G_seqitems_other_law_binary.nodes[n]['heading']
    for n, cnt in top_cited_seqitem_other_law_binary[:25]
]

In [ ]:
Z = make_weighted(G_seqitems)

In [ ]:
for u, v, w in sorted(Z.in_edges('BJNR004810968_OWiG-1968_20180101_000133', data='weight'), key=lambda x: -x[-1]):
    print(f'{w:4.0f} - {u} -', Z.nodes[u]['heading'])